In [1]:
import sqlite3
from pathlib import Path

DB_PATH = Path("..") / "database" / "ans.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("Banco conectado:", DB_PATH.resolve())

Banco conectado: C:\Users\Uélinton\jupyter_projects\ans_market_analytics\database\ans.db


In [2]:
query = """
WITH operadora_mes AS (
    SELECT
        competencia,
        registro_ans,
        MAX(beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora
    GROUP BY competencia, registro_ans
)

SELECT
    competencia,
    SUM(beneficiarios) AS total_beneficiarios
FROM operadora_mes
GROUP BY competencia
ORDER BY competencia;
"""

cursor.execute(query)

rows = cursor.fetchall()

for r in rows:
    print(r)

('2025-12-01', 67990728)


In [3]:
query = """
WITH operadora_mes AS (
    SELECT
        competencia,
        registro_ans,
        MAX(beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora
    GROUP BY competencia, registro_ans
),

setor_mes AS (
    SELECT
        competencia,
        SUM(beneficiarios) AS total_beneficiarios
    FROM operadora_mes
    GROUP BY competencia
),

growth AS (
    SELECT
        competencia,
        total_beneficiarios,
        LAG(total_beneficiarios) OVER (ORDER BY competencia) AS mes_anterior
    FROM setor_mes
)

SELECT
    competencia,
    total_beneficiarios,
    mes_anterior,
    total_beneficiarios - mes_anterior AS crescimento_absoluto
FROM growth
ORDER BY competencia;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('2025-12-01', 67990728, None, None)


In [4]:
query = """
WITH modalidade_mes AS (
    SELECT
        competencia,
        modalidade_grupo,
        MAX(beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora
    GROUP BY
        competencia,
        registro_ans,
        modalidade_grupo
),

total_modalidade AS (
    SELECT
        competencia,
        modalidade_grupo,
        SUM(beneficiarios) AS total_beneficiarios
    FROM modalidade_mes
    GROUP BY
        competencia,
        modalidade_grupo
)

SELECT
    competencia,
    modalidade_grupo,
    total_beneficiarios,

    ROUND(
        total_beneficiarios * 100.0 /
        SUM(total_beneficiarios) OVER (PARTITION BY competencia),
        4
    ) AS market_share_modalidade
FROM total_modalidade
ORDER BY competencia, market_share_modalidade DESC;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('2025-12-01', 'Medicina de Grupo', 22744679, 33.4526)
('2025-12-01', 'Odontologia de Grupo', 15168640, 22.3099)
('2025-12-01', 'Cooperativa Médica', 13221175, 19.4456)
('2025-12-01', 'Seguradora', 7324346, 10.7726)
('2025-12-01', 'Autogestão', 5529960, 8.1334)
('2025-12-01', 'Cooperativa odontológica', 3116606, 4.5839)
('2025-12-01', 'Filantropia', 885322, 1.3021)


In [5]:
query = """
WITH modalidade_mes AS (
    SELECT
        competencia,
        modalidade_grupo,
        MAX(beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora
    GROUP BY
        competencia,
        registro_ans,
        modalidade_grupo
),

total_modalidade AS (
    SELECT
        competencia,
        modalidade_grupo,
        SUM(beneficiarios) AS total_beneficiarios
    FROM modalidade_mes
    GROUP BY
        competencia,
        modalidade_grupo
)

SELECT
    competencia,
    modalidade_grupo,
    total_beneficiarios,

    RANK() OVER (
        PARTITION BY competencia
        ORDER BY total_beneficiarios DESC
    ) AS ranking_modalidade
FROM total_modalidade
ORDER BY competencia, ranking_modalidade;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('2025-12-01', 'Medicina de Grupo', 22744679, 1)
('2025-12-01', 'Odontologia de Grupo', 15168640, 2)
('2025-12-01', 'Cooperativa Médica', 13221175, 3)
('2025-12-01', 'Seguradora', 7324346, 4)
('2025-12-01', 'Autogestão', 5529960, 5)
('2025-12-01', 'Cooperativa odontológica', 3116606, 6)
('2025-12-01', 'Filantropia', 885322, 7)


## Panorama do setor de saúde suplementar

O setor de saúde suplementar brasileiro apresenta forte participação de planos médico-hospitalares, com participação relevante também de planos odontológicos.

A evolução do número total de beneficiários permite observar tendências de crescimento ou retração do mercado ao longo do tempo.

Esse tipo de análise é utilizado por reguladores e analistas de mercado para monitorar a estrutura competitiva do setor.